# Phase 4B — Classical Machine Learning Model Training & Controlled Experimentation

**Project:** Automated Classification of Martian Surface Images Captured by NASA's Curiosity Rover Using Machine Learning

### Overview
This notebook documents the training, validation, and comparative evaluation of five classical machine learning model families across **14 strictly controlled experimental configurations**:
1. **K-Nearest Neighbors (KNN)** (4 configurations: Scaled vs. Scaled+PCA; Uniform vs. Distance weighting)
2. **Gaussian Naive Bayes** (2 configurations: Scaled; Empirical vs. Uniform Class Priors)
3. **Decision Tree** (2 configurations: Unscaled Raw Features; Baseline vs. Balanced Class Weights)
4. **Random Forest** (2 configurations: Unscaled Raw Features; Baseline vs. Balanced Class Weights)
5. **Support Vector Machine (SVM)** (4 configurations: Scaled vs. Scaled+PCA; Baseline vs. Balanced Class Weights)

### Protocol Guarantees
- **Zero Data Leakage:** Preprocessing (`StandardScaler` and `PCA(n_components=100)`) fitted **exclusively** on `X_train`.
- **Zero Dataset Rebalancing:** Natural ~122× class imbalance preserved; no SMOTE, oversampling, undersampling, or synthetic data.
- **Model Selection:** Primary selection metric is **Validation Macro-F1**.
- **Unbiased Generalization:** Exactly one finalized winning model is evaluated once on the untouched Test set.

In [ ]:
import sys
import os
import time
import json
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

import config
print(f'Project root: {config.PROJECT_ROOT}')

## 1. Verify Phase 3 Feature Artifacts
Load the extracted 8,186-dimensional feature representations and official split labels.

In [ ]:
with np.load(config.FEATURES_DIR / 'X_train.npz') as d:
    X_train = d['X']
y_train = np.load(config.FEATURES_DIR / 'y_train.npy')

with np.load(config.FEATURES_DIR / 'X_val.npz') as d:
    X_val = d['X']
y_val = np.load(config.FEATURES_DIR / 'y_val.npy')

with np.load(config.FEATURES_DIR / 'X_test.npz') as d:
    X_test = d['X']
y_test = np.load(config.FEATURES_DIR / 'y_test.npy')

print(f'X_train shape: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_val shape:   {X_val.shape}, y_val:   {y_val.shape}')
print(f'X_test shape:  {X_test.shape}, y_test:  {y_test.shape}')

assert X_train.shape == (3746, 8186)
assert X_val.shape == (1640, 8186)
assert X_test.shape == (1305, 8186)
assert np.isfinite(X_train).all() and np.isfinite(X_val).all() and np.isfinite(X_test).all()
print('[PASS] Phase 3 feature matrices verified (100% finite, exact split dimensions).')

## 2. Preprocessing & PCA Verification
Review the PCA fitted exclusively on `X_train_scaled`. The actual cumulative explained variance from the real training data is recorded.

In [ ]:
with open(config.RESULTS_DIR / 'phase4b_experiment_metadata.json', 'r') as f:
    metadata = json.load(f)

pca_info = metadata['pca_specification']
print(f'PCA Components                  : {pca_info["n_components"]}')
print(f'Fitted Exclusively On           : {pca_info["fitted_exclusively_on"]}')
print(f'Actual Cumulative Explained Var : {pca_info["actual_cumulative_explained_variance"]*100:.2f}%')
print(f'Top-1 Component Explained Var   : {pca_info["actual_top1_explained_variance"]*100:.2f}%')

## 3. Results of the 14 Controlled Experiments
Below is the complete validation performance table across all 14 configurations, sorted by **Validation Macro-F1**.

In [ ]:
df_results = pd.read_csv(config.RESULTS_DIR / 'phase4b_results.csv')
display(df_results[[
    'Config ID', 'Model Name', 'Representation', 'Features',
    'Imbalance Strategy', 'Validation Macro-F1', 'Validation Accuracy (%)',
    'Validation Macro Precision', 'Validation Macro Recall', 'Train Time (s)', 'Val Pred Time (s)'
]])

## 4. Visualizing Validation Macro-F1 Across Models
Macro-F1 is the primary model-selection metric because the dataset features an extreme 122× class imbalance.

In [ ]:
display(Image(filename=str(config.PLOTS_DIR / 'phase4b_macro_f1_comparison.png')))

## 5. Selected Winning Model Configuration
Based strictly on the highest Validation Macro-F1, the winning configuration is identified.

In [ ]:
winner = metadata['selected_winner']
print(f'Winning Configuration ID : {winner["config_id"]}')
print(f'Model Specification      : {winner["model_name"]}')
print(f'Feature Representation   : {winner["representation"]}')
print(f'Imbalance Strategy       : {winner["imbalance_strategy"]}')
print(f'Validation Macro-F1      : {winner["val_macro_f1"]:.4f}')
print(f'Validation Accuracy      : {winner["val_accuracy"]:.2f}%')

## 6. Final Test Set Evaluation (Executed Strictly Once on Winner)
Generalization performance of the finalized winning model on the 1,305 untouched test samples.

In [ ]:
print(f'Final Test Macro-F1        : {winner["final_test_macro_f1"]:.4f}')
print(f'Final Test Accuracy        : {winner["final_test_accuracy"]:.2f}%')
print(f'Final Test Macro Precision : {winner["final_test_macro_precision"]:.4f}')
print(f'Final Test Macro Recall    : {winner["final_test_macro_recall"]:.4f}')
print(f'Final Test Weighted-F1     : {winner["final_test_weighted_f1"]:.4f}')

# Display Test Confusion Matrix
display(Image(filename=str(config.CONFUSION_MATRICES_DIR / f'test_confusion_matrix_{winner["config_id"]}.png')))

## 7. Per-Class Performance and Minority Class Interpretation
> **Methodological Note on Minority Classes:**
> Support numbers must be taken into account when interpreting per-class metrics. For example, `portion tube opening` achieves Precision=1.0, Recall=1.0, and F1=1.0, but only has **2** instances in the test set. Such high performance on very small sample sizes should not be overclaimed as guaranteed real-world generalization.

In [ ]:
df_test_report = pd.read_csv(config.CLASSIFICATION_REPORTS_DIR / f'test_classification_report_{winner["config_id"]}.csv', index_col=0)
display(df_test_report)

## 8. Key Architectural Takeaways

1. **Dimensionality Reduction Impact:**
   - In the high-dimensional 8,186 feature space, distance metrics suffer from the 'curse of dimensionality', leading KNN to achieve only Macro-F1 of 0.5479 (Uniform) and 0.5837 (Distance).
   - Projecting to 100 PCA components (accounting for **67.16%** of variance) drastically boosts KNN performance to **0.6306** (Uniform) and **0.6613** (Distance) while accelerating inference from ~0.40s down to ~0.01s.
   - Similarly, SVM with PCA achieves Macro-F1 of **0.6393** with training time of **0.63s**, compared to 68.95s on the full 8,186 features.

2. **Tree-Based Models on Unscaled Features:**
   - Random Forest with balanced class weights on raw unscaled features demonstrated robust accuracy (66.04%) and Macro-F1 (0.5993).
   - Single Decision Trees overfitted heavily, reaching only 0.2047–0.2690 Macro-F1.

3. **Naive Bayes Feature Correlation:**
   - Gaussian Naive Bayes yielded ~0.20 Macro-F1 due to severe feature correlation in dense 8,100 HOG and color histogram bins violating the conditional independence assumption.

4. **Class Weighting & Distance Weighting:**
   - In KNN, distance-weighted voting (`weights='distance'`) consistently outperformed uniform voting (+3.58% F1 in full space, +3.07% F1 in PCA space) by giving higher influence to nearby minority-class neighbors.